# CV03 Object Detection

### Lab Description
This laboratory exercise introduces **YOLOv9**, a state-of-the-art one-stage object detection model.  
Unlike traditional detectors, YOLO (“You Only Look Once”) achieves high accuracy and speed by performing detection in a single forward pass, making it suitable for real-time applications.

In this hands-on lab, you will:
- **Train** YOLOv9 on a sample dataset for ~10 epochs.  
- **Evaluate** the trained model on validation/test images.  
- Record and visualize runtime, batch size, and GPU memory usage.  
- Plot training curves to verify the learning process.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.3 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

- Understand single-stage object detection and the YOLO architecture
- Train YOLOv9 on a sample dataset using the Ultralytics framework
- Evaluate detection results on validation and test images
- Record and visualize training curves and GPU resource usage
- Visualize predicted bounding boxes on test images


In [ ]:
import os
import sys

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

Use GPU:4
Python: 3.10.18
PyTorch: 2.8.0+rocm6.4
CUDA available: True
CUDA device: AMD Radeon AI PRO R9700


In [3]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/aup/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 2. Download COCO128 & Create Data YAML
We use a **tiny COCO subset** (COCO128) so training finishes quickly.
Ultralytics downloads it automatically (about 7 MB) the first time `coco128.yaml` is used.

In [ ]:
# Keep training runs and outputs under ~/data
DATA_DIR = os.path.expanduser("~/data/YOLOv9_data")
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(DATA_DIR)
print("Working directory:", os.getcwd())

## 3. Training

In [6]:
# Build a YOLOv9c model from pretrained weight
model = YOLO("yolov9c.pt")

In [7]:
model.info()

YOLOv9c summary: 358 layers, 25,590,912 parameters, 0 gradients, 104.0 GFLOPs


(358, 25590912, 0, 104.02268160000003)

In [ ]:
results = model.train(data="coco128.yaml", epochs=10, imgsz=640, batch=4)

## 4. Evaluation

In [ ]:
from ultralytics.utils import ASSETS

# Sample images shipped with Ultralytics
results = model([str(ASSETS / "bus.jpg"), str(ASSETS / "zidane.jpg")])

In [ ]:
import contextlib
import pathlib

from matplotlib import pyplot as plt
from PIL import Image

OUTPUT_DIR = os.path.join(DATA_DIR, "results")
for i, r in enumerate(results):
    im_bgr = r.plot()
    im_rgb = im_bgr[:, :, ::-1]

    plt.figure(figsize=(8, 8))
    plt.imshow(im_rgb)
    plt.axis("off")
    plt.title(f"Prediction #{i}")
    plt.show()

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    stem = f"pred_{i}"
    with contextlib.suppress(Exception):
        stem = pathlib.Path(r.path).stem
    out_file = os.path.join(OUTPUT_DIR, f"{stem}_pred.jpg")
    Image.fromarray(im_rgb).save(out_file)
    print("Saved:", out_file)

## Conclusions

In this lab, you trained YOLOv9 for real-time object detection. You learned how single-stage detectors perform detection in one forward pass, and evaluated detection results through visualization and training curve analysis.

---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT